# Single-Agent Design: A First-Principles Guide

## The Core Idea

The basic algorithm is:

```python
while True:
    action = model.decide(context)

    if should_stop(action):
        break

    # Note: if interaction mode, add user check/whitelist etc before 
    # executing an action.
    result = environment.execute(action)
    
    context = update_context(context, action, result)
```

The actual implementation requires error handling, validation, execution limits, and other controls, but the underlying idea remains simple.

The eight questions are:

1. What is the goal, and what counts as success?
2. What actions can the LLM take?
3. How are those actions interpreted and executed?
4. How is context managed?
5. When and why does the agent stop?
6. How are failures handled?
7. How are actions executed safely?
8. How is agent behavior observed and evaluated?

**The loop is simple. Most agent engineering is about making this loop reliable, efficient, safe, and measurable.**

---

## 0. What Is the Agent Trying to Achieve?

**Core problem:** Before building an agent, we need to define what it should accomplish and how to tell whether it succeeded.

- **Task Definition:** Clearly describe the job the agent must complete.
- **Success Criteria:** Define measurable evidence that the task was completed correctly.
- **Independent Verification:** Check the result using something other than the LLM's own claims, such as hidden tests.

**Example:** A coding agent's task is to fix `median()`. Success means the corrected function passes independent tests, not merely that the LLM says "Done."

## 1. What Actions Can the LLM Take?

**Core problem:** The LLM can generate requests, but it needs external tools to interact with the real world.

### Design Choices

- **Bash Tool:** Give the LLM a general-purpose command interface that can read files, modify code, and run programs.
- **Dedicated Tools:** Provide specialized functions such as `read_file()`, `write_file()`, or `run_tests()`.
- **Tool Permissions:** Restrict which actions and resources each tool can access.
- **Tool Abstraction:** Hide complicated implementation details behind a simple interface.

### Trade-off: Bash vs. Dedicated Tools

**Bash offers flexibility. Dedicated tools offer control.**

For example, a dedicated `update_file()` tool can automatically check whether a file exists, validate its path, and reject unauthorized changes.

With Bash, the LLM may need to construct those checks itself, unless the runtime imposes additional restrictions.

A single Bash tool can perform many tasks, but unrestricted Bash also exposes a much larger and potentially dangerous action space.

## 2. How Do We Interpret and Execute LLM Actions?

**Core problem:** The model produces an action request, but that request must be converted into a valid operation that the program can safely execute.

### Design Choices

- **Tool Schema:** A description of what a tool does, which arguments it accepts, and what values are allowed.
- **Tool Calling:** A structured way for the LLM to request an external operation.
- **JSON Parsing:** Convert the model's JSON-formatted arguments into data that Python can use.
- **Argument Validation:** Check that the arguments are valid before executing the action.
- **Action Dispatching:** Select the correct tool implementation based on the requested tool name.
- **Observation Formatting:** Convert a tool's raw output into a message format that the LLM can read on its next turn.

### Example: Observation Formatting

Suppose a Bash command returns:

```python
{
    "output": "2 tests passed",
    "returncode": 0
}
```

The agent might format it as:

```python
{
    "role": "tool",
    "tool_call_id": "call_123",
    "content": "Exit code: 0\nOutput: 2 tests passed"
}
```

The first dictionary is the **raw execution result**.

The second is the **observation message** sent back to the LLM.

Observation formatting is simply translating between these two representations.

**Key principle: The model proposes actions; the runtime validates and executes them.**

## 3. How Do We Manage Context?

**Core problem:** The LLM has limited context capacity, while an agent may accumulate a very long execution history.

### Design Choices

- **Message History:** Store the conversation, action requests, and tool results.
- **Context Window:** The maximum amount of input and generated information that the model can handle within its context limit.
- **Truncation:** Remove or shorten older information to reduce context usage.
- **Summarization:** Replace a long history with a shorter summary of the important facts.
- **Retrieval:** Fetch relevant information when needed instead of keeping everything in the prompt.
- **Context Selection:** Decide which previous messages and observations are actually necessary for the next decision.

### Example

An agent reads a 2,000-line file.

Should those 2,000 lines be sent to the model on every subsequent turn?

Not necessarily.

The agent might preserve the full file in its execution log but retain only the relevant functions or a summary in the model's context.

**Key distinction: Stored history and model context do not have to be identical.**

## 4. When and How Should the Agent Stop?

**Core problem:** An agent cannot run forever. It needs clear stopping conditions.

### Design Choices

- **Final Answer:** Stop when the LLM returns a final response instead of requesting another tool.
- **Sentinel:** Use a special marker such as `TASK_COMPLETE` to indicate that the task is finished.
- **Exit Event:** Create an internal event that records why the agent is stopping.
- **Maximum Steps:** Stop after a fixed number of model turns or actions.
- **Time Limit:** Stop when execution exceeds an allowed duration.
- **Cost Limit:** Stop when API spending reaches a predefined budget.
- **Human Intervention:** Pause when the agent needs user approval or additional information.

### Example

```python
MAX_STEPS = 10

for step in range(MAX_STEPS):
    response = call_llm()

    if not response.tool_calls:
        break

    execute_tools(response.tool_calls)
```

This has two stopping mechanisms:

1. The model returns no tool calls.
2. The loop reaches its maximum step count.

However, **stopping does not prove success**.

An agent may stop because it believes the task is finished, even when its solution is incorrect.

**Key distinction: Agent Completion ≠ Task Success.**

## 5. What Happens When Something Fails?

**Core problem:** Models generate imperfect actions, external tools fail, and infrastructure can be unreliable.

### Design Choices

- **Timeout:** Stop a tool or API operation that takes too long.
- **Invalid Arguments:** Reject tool requests with missing or malformed parameters.
- **Execution Error:** Detect when a command fails or returns a nonzero exit code.
- **Retry:** Repeat an operation when a temporary failure may disappear.
- **Error Feedback:** Send failure information back to the LLM so it can revise its approach.
- **Failure Limits:** Stop when repeated errors suggest that further attempts are unlikely to help.
- **Recovery Strategy:** Decide whether to retry, modify the action, use an alternative tool, or terminate.

### Example

The LLM requests:

```bash
python test.py
```

The environment returns:

```text
ModuleNotFoundError: No module named 'pytest'
```

The agent can send this error back to the LLM.

The LLM may then choose another approach.

However, if it runs the same failing command 20 times, the runtime should eventually intervene.

### Important Failure Categories

- **Task Failure:** The code or proposed solution is wrong; the agent may be able to fix it.
- **Tool Failure:** An operation fails or receives invalid parameters.
- **Infrastructure Failure:** The API, network, or execution environment becomes unavailable.

These failures may require different recovery strategies.

## 6. How Do We Execute Actions Safely?

**Core problem:** Giving an LLM the ability to execute code or modify files creates real-world risks.

### Design Choices

- **Local Execution:** Execute commands directly on the host computer, using the permissions available to the process.
- **Sandbox:** Run potentially untrusted operations inside a restricted environment.
- **Docker Container:** Package an execution environment into an isolated container with configurable resource and permission boundaries.
- **Permissions:** Restrict access to specific files, directories, commands, and external services.
- **Human Approval:** Require explicit permission before performing sensitive or irreversible actions.
- **Resource Limits:** Restrict CPU, memory, execution time, and other resources.
- **Input Validation:** Reject unsafe or unauthorized requests before they reach the execution environment.

### Example

An agent needs to fix `solution.py`.

It should not automatically gain permission to modify every file on your computer.

A dedicated tool can restrict writes to `solution.py`.

Alternatively, a sandboxed Bash environment can limit the agent to a temporary workspace.

**Key principle: Never rely on the LLM's willingness to follow instructions as the only security boundary.**

Security restrictions must be enforced by the runtime.

## 7. How Do We Observe and Evaluate the Agent?

**Core problem:** Without a record of execution, we cannot reliably understand failures, compare experiments, or improve the agent.

### Design Choices

- **Logging:** Record what happened during execution.
- **Tracing:** Preserve the ordered sequence of model decisions, tool calls, and tool results.
- **Serialization:** Convert runtime information into a storable representation, such as a JSON-compatible dictionary.
- **Cost Tracking:** Measure input tokens, output tokens, API expenses, and other resource usage.
- **Evals:** Measure how successfully and reliably an agent completes tasks.
- **Trajectory Inspection:** Provide a readable interface for investigating an agent's execution history.
- **Experiment Tracking:** Associate results with the model, prompts, tools, configuration, and test conditions used.

### Example: Our Coding Agent

Suppose one run produces:

- 3 model turns.
- 15 tool calls.
- 7 repeated file reads.
- 6,523 total tokens.
- All public tests passing.

What can we conclude?

**Logging** tells us that 15 tools were called.

**Tracing** shows when the repeated reads occurred and what happened afterward.

**Cost Tracking** tells us how expensive the run was.

**Evals** tell us whether the final solution actually works.

### Important Distinction

**Logging answers:** What happened?

**Tracing answers:** In what order did things happen, and how are the events related?

**Evals answer:** Did the agent perform well?

These are related, but they solve different problems.

---

## Additional Concerns for Production Agents

The eight questions above cover the main design decisions of a basic single agent.

Larger systems may need additional engineering.

### Persistence and Checkpointing

**Persistence:** Store information so it survives process restarts.

**Checkpointing:** Save enough execution state to resume an interrupted task without starting from scratch.

Serialization alone does not guarantee resumability.

### Concurrency

**Concurrency:** Manage multiple tasks, agents, or tool operations that can make progress during overlapping periods of time.

This becomes important when several agents share files, databases, budgets, or external services.

### Monitoring

**Monitoring:** Continuously track the health of running agents and detect unusual behavior, errors, excessive spending, or performance degradation.

Unlike debugging a single execution trace, monitoring operates across many runs.

### Versioning and Reproducibility

**Versioning:** Record which model, prompts, tools, and software versions produced an outcome.

**Reproducibility:** Make experiments sufficiently controlled and documented that results can be investigated and compared.

LLM outputs can vary between runs, even with similar settings.
